# Does `aoutools` lose homozygous-reference samples on the non-split path?

Run this on the **All of Us Researcher Workbench**, on a Hail Genomic Analysis
environment. It is deliberately cheap: one small window of chr1, a few hundred
samples, a handful of variants.

## The claim under test

A VDS stores homozygous-reference calls **sparsely**: `variant_data` is supposed to
hold entries only for samples with a non-reference call, and hom-ref samples live in
`reference_data` as blocks. If that is true, then a hom-ref sample is *absent* from
`variant_data` — and Hail's entry aggregators skip absent entries entirely.

That would mean `_calculate_dosage`'s opening branch —

```python
hl.case()
  .when(hl.is_missing(alleles_expr) & ref_is_effect, 2)   # "sparse hom-ref"
  .when(hl.is_missing(alleles_expr) & ~ref_is_effect, 0)
```

— is **never reached** by the samples it was written for, because they are not in the
stream to be evaluated. And that in turn would mean:

> **H1.** With `split_multi=False`, a weights row whose **effect allele is the REF**
> scores every hom-ref sample as **0 copies** when the truth is **2 copies**.
> Carriers are scored correctly. So the error hits only some samples — it is
> genotype-dependent, and it **changes the ranking**.
>
> **H2.** With `split_multi=True, ref_is_effect_allele=True`, the score is short by
> `2w` per matched variant — but by the **same** `2w` for *every* sample. Absolute
> scores shift; rankings survive.

## How this notebook could prove me wrong

Everything is checked against an **independent ground truth**: `hl.vds.to_dense_mt`
reconstitutes the full genotype matrix from the reference blocks, so every sample has
a real genotype and copies of the effect allele can be counted directly. If the
library's score matches that ground truth, **H1 is false and I was wrong.**

Cell 8 prints a verdict either way. It does not assume the answer.

In [ ]:
# If aoutools isn't already installed in this environment:
# !pip install -q git+https://github.com/dokyoonkimlab/aoutools.git@dev

import os

import hail as hl
import pandas as pd

from aoutools.prs._calculator import _calculate_prs_chunk
from aoutools.prs._calculator_utils import _validate_and_prepare_weights_table
from aoutools.prs._config import PRSConfig

hl.init(default_reference="GRCh38", idempotent=True)

VDS_PATH = os.getenv("WGS_VDS_PATH")
print("VDS:", VDS_PATH)

In [ ]:
# Keep this small. A narrow window and a few hundred samples is enough, and the
# whole notebook then costs a couple of minutes of cluster time.
INTERVAL = "chr1:55039447-55064852"  # PCSK9; any gene-sized window works
N_SAMPLES = 200
N_VARIANTS = 5

vds = hl.vds.read_vds(VDS_PATH)

vds = hl.vds.filter_intervals(
    vds,
    [hl.parse_locus_interval(INTERVAL, reference_genome="GRCh38")],
    keep=True,
)

sample_ids = vds.variant_data.cols().head(N_SAMPLES).s.collect()
samples_ht = hl.Table.parallelize(
    [{"s": s} for s in sample_ids], hl.tstruct(s=hl.tstr)
).key_by("s")
vds = hl.vds.filter_samples(vds, samples_ht)

print(f"{len(sample_ids)} samples, window {INTERVAL}")

In [ ]:
# Pick real biallelic SNPs from the VDS itself, rather than hardcoding rsIDs, so
# the weights are guaranteed to match variants that actually exist.
rows = vds.variant_data.rows()
rows = rows.filter(
    (hl.len(rows.alleles) == 2)
    & (hl.len(rows.alleles[0]) == 1)
    & (hl.len(rows.alleles[1]) == 1)
)
variants = rows.head(N_VARIANTS).select().collect()

VARIANTS = [
    {"contig": v.locus.contig, "pos": v.locus.position,
     "ref": v.alleles[0], "alt": v.alleles[1]}
    for v in variants
]
K = len(VARIANTS)

# Restrict the VDS to exactly these K variants, so nothing else can contribute.
keep = hl.Table.parallelize(
    [{"locus": hl.Locus(v["contig"], v["pos"], reference_genome="GRCh38")}
     for v in VARIANTS],
    hl.tstruct(locus=hl.tlocus("GRCh38")),
).key_by("locus")
intervals = keep.select(
    interval=hl.interval(keep.locus, keep.locus, includes_end=True)
).key_by("interval")
vds = hl.vds.filter_intervals(vds, intervals, keep=True)

pd.DataFrame(VARIANTS)

## Experiment 1 — is a hom-ref sample present in `variant_data` at all?

Two counts per sample:

* `n_entries_variant_data` — how many of the K rows Hail's aggregator actually
  **visits** for this sample in `variant_data`.
* `n_entries_dense` — the same count after `to_dense_mt`, where hom-ref calls have
  been reconstituted from the reference blocks. This should always be K.

If sparse storage works the way the claim assumes, a sample that is hom-ref at all K
sites will show **0** in the first column and **K** in the second.

In [ ]:
vd = vds.variant_data

sparse_counts = (
    vd.select_cols(n_entries_variant_data=hl.agg.count()).cols().to_pandas()
)

dense = hl.vds.to_dense_mt(vds)
dense = dense.annotate_entries(GT=hl.vds.lgt_to_gt(dense.LGT, dense.LA))
dense_counts = (
    dense.select_cols(n_entries_dense=hl.agg.count()).cols().to_pandas()
)

counts = sparse_counts.merge(dense_counts, on="s")
print(f"K = {K} variants\n")
print(counts["n_entries_variant_data"].value_counts().sort_index())
print()
print(
    "samples visited 0 times in variant_data:",
    int((counts.n_entries_variant_data == 0).sum()),
)
print(
    "...but present in the dense matrix:",
    int(
        (
            (counts.n_entries_variant_data == 0)
            & (counts.n_entries_dense == K)
        ).sum()
    ),
)
counts.head(10)

## Experiment 2 — the independent ground truth

From the **dense** matrix, count each sample's true copies of the effect allele.
The effect allele is chosen to be the **REF** base at each of the K variants, and
every weight is `1.0`, so a sample's true score is literally its total number of
reference-allele copies across the K sites. Nothing in this cell touches `aoutools`.

In [ ]:
# The mock GWAS summary: effect allele = REF, noneffect = ALT, weight = 1.0.
WEIGHTS = [
    {"chr": v["contig"], "pos": v["pos"],
     "effect_allele": v["ref"], "noneffect_allele": v["alt"], "weight": 1.0}
    for v in VARIANTS
]
raw_weights = hl.Table.parallelize(
    WEIGHTS,
    hl.tstruct(chr=hl.tstr, pos=hl.tint32, effect_allele=hl.tstr,
               noneffect_allele=hl.tstr, weight=hl.tfloat64),
)

# Ground truth: count copies of the REF allele in the DENSE matrix, where every
# sample has a real genotype.
gt_alleles = hl.array([dense.alleles[dense.GT[0]], dense.alleles[dense.GT[1]]])
ref = dense.alleles[0]
truth_mt = dense.annotate_entries(
    true_copies=hl.or_else(
        gt_alleles.filter(lambda a: a == ref).length(), 0
    ),
    is_hom_ref=hl.or_else(dense.GT.is_hom_ref(), False),
)
truth = (
    truth_mt.select_cols(
        truth=hl.agg.sum(truth_mt.true_copies),
        n_hom_ref=hl.agg.count_where(truth_mt.is_hom_ref),
    )
    .cols()
    .to_pandas()
)
truth.head()

## Experiment 3 — what does `aoutools` actually score?

Both configurations, through the library's own code path.

In [ ]:
def library_score(**cfg):
    config = PRSConfig(include_n_matched=True, **cfg)
    wt = _validate_and_prepare_weights_table(raw_weights, config)
    df = _calculate_prs_chunk(wt, vds, config).to_pandas()
    n_matched = int(df["n_matched"].iloc[0])
    return df.rename(columns={"person_id": "s"})[["s", "prs"]], n_matched


non_split, n1 = library_score(split_multi=False, strict_allele_match=True)
split_ref, n2 = library_score(split_multi=True, ref_is_effect_allele=True)

print(f"variants matched -- non-split: {n1}/{K},  split: {n2}/{K}")
assert n1 == K and n2 == K, "a variant failed to match; the comparison is void"

res = (
    truth.merge(non_split.rename(columns={"prs": "non_split"}), on="s")
    .merge(split_ref.rename(columns={"prs": "split_ref"}), on="s")
)
res["err_non_split"] = res["truth"] - res["non_split"]
res["err_split"] = res["truth"] - res["split_ref"]
res.sort_values("n_hom_ref").head(12)

## Verdict

Read `err_non_split` and `err_split` above.

* **H1 holds** if `err_non_split == 2 * n_hom_ref` — the error grows with how many
  sites the sample is hom-ref at, so it differs between samples and reorders them.
* **H2 holds** if `err_split == 2 * K` for *every* sample — one constant, no
  reordering. (`w = 1`, K matched variants, hence `2w × K`.)
* **I am wrong** if `err_non_split == 0`, i.e. the library already matches the dense
  ground truth.

The cell below decides it.

In [ ]:
h1 = bool((res.err_non_split == 2 * res.n_hom_ref).all())
h1_null = bool((res.err_non_split == 0).all())
h2 = bool((res.err_split == 2 * K).all())

# Does the non-split error actually reorder people, or is it a harmless shift?
reorders = (
    res[["truth", "non_split"]].corr(method="spearman").iloc[0, 1] < 1.0
)

print(f"non-split error == 2 * n_hom_ref  (H1) : {h1}")
print(f"non-split error == 0  (I was wrong)    : {h1_null}")
print(f"split error == 2*K, same for everyone   : {h2}")
print(f"distinct non-split error values         : "
      f"{sorted(res.err_non_split.unique())}")
print(f"distinct split error values             : "
      f"{sorted(res.err_split.unique())}")
print(f"non-split changes the RANKING           : {reorders}")
print()

if h1_null:
    print("VERDICT: the claim is FALSE. aoutools matches the dense ground "
          "truth; hom-ref samples are scored correctly.")
elif h1 and h2:
    print("VERDICT: the claim is CONFIRMED on the real All of Us VDS.\n"
          "  - non-split loses 2 copies for every site a sample is hom-ref at,\n"
          "    which varies per sample and therefore reorders people.\n"
          f"  - split is uniformly short by 2w*K = {2 * K} for everyone, so\n"
          "    absolute scores shift but rankings are preserved.")
else:
    print("VERDICT: neither hypothesis fits. Inspect `res` -- the real "
          "behaviour is something else again.")

## Side check — do no-call genotypes even exist in `variant_data`?

A separate claim: an entry that *exists* but whose genotype is missing (a genuine
no-call) hits the same missing-genotype branch and is handed **2 copies** of the
reference. That only matters if AoU's `variant_data` actually contains such entries.
If the count below is 0, that bug is theoretical.

In [ ]:
n_entries = vd.aggregate_entries(hl.agg.count())
n_nocall = vd.aggregate_entries(hl.agg.count_where(hl.is_missing(vd.LGT)))
print(f"entries in variant_data : {n_entries}")
print(f"...with a missing LGT   : {n_nocall}")
print(
    "\nno-call bug is REAL here" if n_nocall else
    "\nno-call bug is theoretical in this window"
)